# Week5_ex2 - Linear Motor Force (1 Slot)

1-slot version of a linear motor: a magnet array on the mover and a single coil/slot on the stator, checking the force on the mover with a virtual force calculation.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [ ]:
# 1. Launch AEDT
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week5_ex2"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week5_ex2"

In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
depth = 200     # motor stack depth
M3D["depth"] = f"{depth}mm"

num_pole = 4    # number of magnet poles
M3D["num_pole"] = f"{num_pole}"

m_w = 20   # magnet width
M3D["m_w"] = f"{m_w}mm"

m_h = 3   # magnet height
M3D["m_h"] = f"{m_h}mm"

gap_w = m_w * 0.2   # gap between adjacent magnets
M3D["gap_w"] = f"{gap_w}mm"

my_th = 15   # magnet yoke plate thickness
M3D["my_th"] = f"{my_th}mm"

num_slot = 1   # number of slots
M3D["num_slot"] = f"{num_slot}"

airgap = 5     # air gap length
M3D["airgap"] = f"{airgap}mm"

t_w = (m_w + gap_w) * 2 / 3   # tooth width
M3D["t_w"] = f"{t_w}mm"

J_rated = 3   # current density in the coil region
M3D["J_rated"] = f"{J_rated}"

cy_th = 15   # yoke thickness
M3D["cy_th"] = f"{cy_th}mm"

c_w = 4   # coil width
M3D["c_w"] = f"{c_w}mm"

c_h = 10   # coil height
M3D["c_h"] = f"{c_h}mm"

t_h = c_h   # tooth height
M3D["t_h"] = f"{t_h}mm"

my_w = (m_w * num_pole + gap_w * (num_pole - 1)) / 3 * 4   # magnet yoke plate width
M3D["my_w"] = f"{my_w}mm"


# material selection
stator_material = 0     # 0 = air, 1 = steel


# current calculation

turn_c = 100    # number of coil turns
M3D["turn_c"] = f"{turn_c}"

area = c_w*c_h      # terminal cross-section area

current_rated = J_rated*area/turn_c     # rated phase current
M3D["current_rated"] = f"{current_rated}A"

In [ ]:
steel = M3D.materials.duplicate_material("steel_1008", name="steel")  # iron core material


magnet_n = M3D.materials.duplicate_material("NdFe35", name="magnet_n")  # magnet with N pole facing +z
magnet_n.get_magnetic_coercivity()  # check current magnetization
magnet_n.set_magnetic_coercivity(value=-890000, x=0, y=0, z=1)  # magnetized along +z


magnet_s = M3D.materials.duplicate_material("NdFe35", name="magnet_s")  # magnet with S pole facing +z (magnetized -z)
magnet_s.get_magnetic_coercivity()  # check current magnetization
magnet_s.set_magnetic_coercivity(value=-890000, x=0, y=0, z=-1)  # magnetized along -z


copper_c = M3D.materials.duplicate_material("copper", name="copper_c")  # copper for the coil

In [ ]:
# Magnetostatic setup, nonlinear BH curve mode on
my_setup = M3D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 20
my_setup.props["MinimumPasses"] = 2
my_setup.props["MinimumConvergedPasses"] = 1
my_setup.props["PercentRefinement"] = 15
my_setup.props["PercentError"] = 1
my_setup.props["MuOption"] = {"MuNonLinearBH": True}
my_setup.update()

In [ ]:
# 1. Magnet yoke plate (substrate)
origin = ["-my_w/2", "-depth/2", "-my_th/2"]
sizes = ["my_w", "depth", "my_th"]
substrate = M3D.modeler.create_box(origin=origin, sizes=sizes, name="substrate", material=steel)


# 2. Magnets
origin = [0, "-depth/2", "-m_h/2"]
sizes = ["m_w", "depth", "m_h"]
magnet_1 = M3D.modeler.create_box(origin=origin, sizes=sizes, name="magnet_1", material=magnet_n)

vector = ["2*(m_w+gap_w)", 0, 0]
M3D.modeler.duplicate_along_line(assignment=magnet_1, vector=vector, clones=num_pole/2, attach=True, is_3d_comp=False, duplicate_assignment=True)   # duplicate along the array

magnet_1.color = (255, 0, 0)    # red


origin = ["gap_w+m_w", "-depth/2", "-m_h/2"]
sizes = ["m_w", "depth", "m_h"]
magnet_2 = M3D.modeler.create_box(origin=origin, sizes=sizes, name="magnet_2", material=magnet_s)

vector = ["2*(m_w+gap_w)", 0, 0]
M3D.modeler.duplicate_along_line(assignment=magnet_2, vector=vector, clones=num_pole/2, attach=True, is_3d_comp=False, duplicate_assignment=True)   # duplicate along the array

magnet_2.color = (0, 0, 255)   # blue


vector = ["-( m_w*num_pole+gap_w*(num_pole-1) )/2", 0, "my_th/2+m_h/2"]
M3D.modeler.move(assignment=[magnet_1, magnet_2], vector=vector)    # center the array and lift it onto the yoke


# 3. Slot
origin = ["-t_w/2", "-depth/2", "-(cy_th+t_h)/2"]
sizes = ["t_w", "depth", "cy_th+t_h"]

if stator_material == 0 :
    slot = M3D.modeler.create_box(origin=origin, sizes=sizes, name="slot", material="air")
    
elif stator_material == 1:
    slot = M3D.modeler.create_box(origin=origin, sizes=sizes, name="slot", material="steel")


# 4. Coil
origin = ["-t_w/2-c_w", "-depth/2-c_w", "-(cy_th+t_h)/2"]
sizes = ["t_w+2*c_w", "depth+2*c_w", "c_h"]
coil = M3D.modeler.create_box(origin=origin, sizes=sizes, name="coil", material=copper_c)

M3D.modeler.subtract(blank_list=coil, tool_list=slot, keep_originals=True)  # coil minus slot

vector = [0, 0, "my_th/2+m_h+airgap+(cy_th+t_h)/2"]
M3D.modeler.move(assignment=[coil, slot], vector=vector)    # place above the magnet array with the air gap


# 5. Region
region = M3D.modeler.create_region(pad_value=["my_w*5", "-my_w*5", "my_w*5", "-my_w*5", "my_w*5", "-my_w*5"], pad_type='Absolute Position', name='Region')


# 6. Insulation boundary between coil and slot
M3D.assign_insulating(assignment=coil, insulation=None)     # without this, slot and coil end up sharing one conduction path

In [ ]:
# 1. Turn one terminal face of the coil into the coil boundary
M3D.modeler.section(assignment=coil, plane="ZX", create_new=True, section_cross_object=False)
tmp = M3D.modeler.sheet_objects[-1]

M3D.modeler.split(assignment=tmp, plane="YZ", sides='PositiveOnly', tool=None, split_crossing_objs=False, delete_invalid_objs=True)
coil1_ter = M3D.modeler.sheet_objects[-1]
    
coil1_terminal = M3D.assign_coil(assignment=coil1_ter, conductors_number="turn_c", polarity='Negative', name="coil1_terminal")


# 2. Create a winding and add the coil to it
coil1 = M3D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current=current_rated, resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="coil1")

M3D.add_winding_coils(assignment=coil1.name, coils=[coil1_terminal.name])   # add the coil to the winding

In [ ]:
# NOTE: MaxLength coarsened from depth/32 to depth/10, and PercentRefinement
# dropped from 30 to 15 -- at the original settings this pushed the mesh to
# ~70,381 elements, over the Student license's cap. Coarser mesh + slower
# per-pass refinement keeps it under the cap while still smoothing the field plot.

oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
    [
		"NAME:Length2",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["coil", "slot"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "depth/10"
	])

In [ ]:
force_name = "force_mover"
M3D.assign_force(assignment=[coil, slot], coordinate_system='Global', is_virtual=True, force_name=force_name)

# M3D.assign_force(assignment=[coil], coordinate_system='Global', is_virtual=True, force_name="force_coil_vir")
# M3D.assign_force(assignment=[coil], coordinate_system='Global', is_virtual=False, force_name="force_coil_lorenz")
# M3D.assign_force(assignment=[slot], coordinate_system='Global', is_virtual=True, force_name="force_slot_vir")
# M3D.assign_force(assignment=[slot], coordinate_system='Global', is_virtual=False, force_name="force_slot_lorenz")

# M3D.assign_force(assignment=[substrate, magnet_1, magnet_2], coordinate_system='Global', is_virtual=True, force_name="force_sub")
# M3D.assign_force(assignment=[magnet_1, magnet_2], coordinate_system='Global', is_virtual=True, force_name="force_mag")
# M3D.assign_force(assignment=[substrate], coordinate_system='Global', is_virtual=True, force_name="force_plate")

In [ ]:
my_setup.analyze()

In [ ]:
## Data table plot ##

oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport("Force Table 1", "Magnetostatic", "Data Table", "Setup1 : LastAdaptive", [],
	[
		"depth:="		, ["All"],
		"num_pole:="		, ["Nominal"],
		"m_w:="			, ["Nominal"],
		"m_h:="			, ["Nominal"],
		"gap_w:="		, ["Nominal"],
		"my_th:="		, ["Nominal"],
		"num_slot:="		, ["Nominal"],
		"airgap:="		, ["Nominal"],
		"t_w:="			, ["Nominal"],
		"J_rated:="		, ["Nominal"],
		"cy_th:="		, ["Nominal"],
		"c_w:="			, ["Nominal"],
		"c_h:="			, ["Nominal"],
		"t_h:="			, ["Nominal"],
		"my_w:="		, ["Nominal"],
		"turn_c:="		, ["Nominal"],
		"current_rated:="	, ["Nominal"]
	],
	[
		"Y Component:="		, ["force_mover.Force_x","force_mover.Force_y","force_mover.Force_z"]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers",
				"Force Table 1:force_mover.Force_x:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers",
				"Force Table 1:force_mover.Force_y:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers",
				"Force Table 1:force_mover.Force_z:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])

In [ ]:
## Save project ##

M3D.save_project()